# Day 5: Gold products, AI/BI and final submission

**Modern Data Engineering for AI Systems · SDA-DSC-214**

Build eight serving products from your approved data. Reconcile reporting totals, inspect time-appropriate AI features, preserve unknown future labels, and submit the project you have built across five days.

**Your lab:** 45 minutes to run and check, 5 minutes to answer, 5 minutes to discuss, and 5 minutes to save.

[Today's guide](https://github.com/almiyead-rgb/masar-ai-data-engineering-ksu/blob/main/day05/README.md) · [Expected results](https://github.com/almiyead-rgb/masar-ai-data-engineering-ksu/tree/main/expected) · [Submission guide](https://github.com/almiyead-rgb/masar-ai-data-engineering-ksu/blob/main/submission/README.md)

## Before you run

Use a CPU runtime. Select the documented Python 3.12 Colab runtime before running the first cell. The setup checks the versions and prepares Java 17. If Colab asks to restart after package installation, reconnect and run the setup again. Keep one GitHub repository for all five days.

Run cells in order. A dependency check is not a completed lab. Compare the outputs produced below with the reference results after the real engines run.

In [ ]:
from pathlib import Path
import importlib.util, json, os, runpy, subprocess, sys
DAY = 5
# Keep your own fork's owner/name here if you want to load committed edits.
PROJECT_REPOSITORY = "almiyead-rgb/masar-ai-data-engineering-ksu"
NONINTERACTIVE = os.environ.get("MASAR_NONINTERACTIVE") == "1"
try:
    IS_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IS_COLAB = False
if IS_COLAB and not NONINTERACTIVE:
    ROOT = Path("/content/masar-project")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/" + PROJECT_REPOSITORY + ".git", str(ROOT)], check=True)
    if not (ROOT / "course.json").is_file():
        raise RuntimeError("The project folder is incomplete. Preserve it and inspect the setup guide.")
    os.chdir(ROOT)
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "course.json").is_file() and (p / "src/masar").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Open this notebook from the complete course repository.")
sys.path.insert(0, str(ROOT / "src"))
bootstrap = runpy.run_path(str(ROOT / "scripts/bootstrap_colab.py"))
environment = bootstrap["prepare"](install=IS_COLAB)
os.environ["JAVA_HOME"] = environment["java_home"]
print(json.dumps(environment, indent=2))
from masar.runtime import start_spark
from masar.workspace import require_fixed_dataset, completed_bronze_workspace, new_workspace, write_json
from masar.native_contracts import validate_stage_result
from masar.colab import restore_handoff, record_reflection, record_metrics, save_day, initialize_student_notes
SOURCE = ROOT / "data/masar-small-v1"
require_fixed_dataset(SOURCE)
print("Course source:", ROOT)
print("Dependency checks are complete. The following cells execute the real engines.")
print("Interactive Drive/download UI enabled:", IS_COLAB and not NONINTERACTIVE)


## Your saved project

Allow Colab to mount your own Drive. Your daily continuation archives are saved in `MyDrive/Masar_Project`. A new session restores the preceding day only when the local project has no saved workspace. Existing work is not overwritten.

If you uploaded a handoff ZIP manually, call `restore_handoff` with its path and the preceding day number. Keep earlier archives while you learn.

After restoration, this cell creates your lab-note prompts under `student/dayNN/` and the shared project notes under `student/`. Open the printed paths in Colab’s Files panel and add your own evidence. Existing files are preserved. Your daily question answer belongs in `student/dayNN/REFLECTION.md`; the final save cell includes all these files in both packages.

In [ ]:
DRIVE_FOLDER = None
if IS_COLAB and not NONINTERACTIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_FOLDER = Path("/content/drive/MyDrive/Masar_Project")
    DRIVE_FOLDER.mkdir(parents=True, exist_ok=True)
if DAY > 1:
    pointer = ROOT / "outputs/day01_bronze_success.json"
    if not pointer.exists():
        backup = DRIVE_FOLDER / f"day{DAY-1:02d}_handoff.zip" if DRIVE_FOLDER else None
        if backup is None or not backup.is_file():
            raise FileNotFoundError(
                f"Restore your day{DAY-1:02d}_handoff.zip before continuing. "
                "Use restore_handoff(ROOT, Path('/path/to/your/archive.zip'), expected_day=DAY-1).")
        print(json.dumps(restore_handoff(ROOT, backup, expected_day=DAY-1), indent=2))
    WORK = completed_bronze_workspace(ROOT)
    print("Continue your project:", WORK.relative_to(ROOT))
else:
    pointer = ROOT / "outputs/day01_bronze_success.json"
    if pointer.exists():
        existing = completed_bronze_workspace(ROOT)
        if (existing / "reports/day02_silver.json").exists():
            raise RuntimeError("Your project has advanced beyond Day 1. Open the next day's notebook. Use a separate clean project for a complete replay.")
        WORK = existing
        REUSE_BRONZE = True
        print("Reusing your completed Bronze workspace without re-ingesting the same batches.")
    else:
        WORK = new_workspace(ROOT, "day01_bronze")
        REUSE_BRONZE = False
        print("New project:", WORK.relative_to(ROOT))

# Create prompts only after restoring your existing work. Never overwrite a note.
notes = initialize_student_notes(ROOT, DAY)
print(json.dumps(notes, indent=2))


## Five-minute activity

Which input was available at a decision cutoff: completed trips received before the cutoff, or the actual trip count in the following hour?

<details><summary>Discuss the answer after trying</summary>

The past trips are eligible if their source revision was available by the cutoff. The following-hour outcome is future information and must remain separate.

</details>

## Lab 07: build a validated serving release

Build eight small Delta products, introduce a controlled candidate failure, and check that the usable release remains available. The recovery exercise then creates another valid release with the same business contents.

In [ ]:
from masar.serving import run_recovery_exercise
spark = start_spark(WORK, kafka=False)
try:
    result = run_recovery_exercise(spark, SOURCE, WORK)
    validate_stage_result('lab07_gold_recovery', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
finally:
    spark.stop()

## Lab 08: consume BI and AI outputs

Run the reporting query, reconcile trip counts and fares, and inspect feature availability. The labels for the unobserved future remain NULL. A `NULL` outcome does not mean a measured outcome of zero.

Your task: inspect `completed_trips_24h` as a historical feature and explain why the future `target_trip_count` cannot be used as an input at the decision cutoff.

In [ ]:
from masar.serving import run_serving_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_serving_lab(spark, SOURCE, WORK)
    validate_stage_result('lab08_serving', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print('BI totals:', json.dumps(result['bi_summary'], indent=2))
    from masar.serving import read_release
    _, observed_tables = read_release(spark, WORK)
    print('AI feature example:', observed_tables['ai.zone_hourly_features'][0])
    print('Future label example:', observed_tables['ai.zone_hourly_labels'][0])
    from decimal import Decimal
    feature_name = "completed_trips_24h"
    print("Selected historical feature:", [row[feature_name] for row in observed_tables["ai.zone_hourly_features"]])
    facts = observed_tables["bi.fact_trips"]
    metrics = {"table_counts": {name: len(rows) for name, rows in observed_tables.items()},
        "trip_count": len(facts), "total_fare_sar": format(sum(Decimal(str(row["fare_sar"])) for row in facts), ".2f"),
        "gps_events": sum(row["gps_event_count"] for row in facts),
        "unobserved_labels": sum(row["label_status"] == "UNOBSERVED" and row["target_trip_count"] is None
                                 for row in observed_tables["ai.zone_hourly_labels"])}
finally:
    spark.stop()

record_metrics(ROOT, DAY, metrics)

## End-of-lab question

Your BI fact table has 75 rows, while the AI feature table has three. Does this show that 72 trips were lost? Explain what a row represents in each table.

Write your answer and one observed value before opening the model answer. Your reflection is part of the project, worth one point when it explains your own run.

In [ ]:
MY_ANSWER = ""  # Write your own explanation before viewing the model answer.
MY_EVIDENCE = ""  # Include a value you observed and the report or cell that produced it.
reflection_path = record_reflection(ROOT, DAY, 'Your BI fact table has 75 rows, while the AI feature table has three. Does this show that 72 trips were lost? Explain what a row represents in each table.', MY_ANSWER, MY_EVIDENCE)
print("Your reflection:", reflection_path.relative_to(ROOT))
if not MY_ANSWER.strip():
    print("Reflection pending: complete the answer in the file before submitting.")

<details><summary>Model answer: open after your attempt</summary>

The tables have different grains. The fact table has one row per trip. A feature row represents one city, decision cutoff and prediction horizon using the eligible historical window. Three feature rows do not imply data loss.

Keep your first answer. Add a short correction in your reflection file if discussion changed your understanding.

</details>

## Save this day

1. Run the save cell below after all Spark sessions have stopped. It creates two packages.
2. In Colab use **File > Save a copy in GitHub**, select your repository and save this notebook as `day05/STUDENT.ipynb`. Keep your outputs.
3. Download the small `day05_github.zip`, unzip it on your computer, and upload its `student` and updated `day02/dbt` files to the same repository. Use a commit message such as `Complete Day 5`.
4. Keep the full handoff ZIP on Drive for your next session. It includes all cumulative Delta files and the separate Day 2 dbt workspace.
5. Open GitHub and confirm the notebook, reflection and results are visible.

Saving a notebook does not save Colab runtime files. The helper never publishes automatically and never asks you to put a token in a cell.

Before this cell, save your edits to the lab notes, shared project notes and reflection. Files marked **PENDING** still need your own explanation and observed evidence. Do not submit the unchanged prompts as completed work.

In [ ]:
saved = save_day(ROOT, DAY, drive_folder=DRIVE_FOLDER)
print(json.dumps(saved, indent=2))
print("Next: save this executed notebook to your GitHub repository using the File menu.")
if IS_COLAB and not NONINTERACTIVE:
    from google.colab import files
    files.download(saved["github_bundle"])

## Final submission

Submit the same repository you updated throughout the course. Include its branch and final commit, the five executed notebooks, your reflections, SQL and evidence files, and the final README. Keep the handoff backup accessible through the agreed private channel when requested.

The project is graded out of **100**. **Attendance on all five days and a project score of at least 70** are required for completion. Full attendance without a project, or with a score below 70, receives an attendance certificate only.